# 프로젝트 2 — Weekend 2: 리랭킹과 답변 품질 평가

**프로젝트**: 검색형 RAG 기반 금융 상품(ETF) 추천 시스템

**이번 주 목표**:
1. LLM 리랭킹으로 검색 결과 품질 향상
2. BLEU, ROUGE, BERTScore 자동 평가 지표 구현
3. LLM-as-Judge 평가 파이프라인 구축
4. Criteria 기반 다차원 평가와 품질 게이트 구현

---
## 환경 설정

In [ ]:
# 환경 설정
import os, json, time, re, math
import numpy as np
import pandas as pd
from collections import Counter
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
MODEL = "gpt-4o-mini"

print("✅ 환경 설정 완료")

## 데이터 준비

In [ ]:
# ETF 샘플 데이터
SAMPLE_ETF_DATA = [
    {"ticker": "KODEX200", "name": "KODEX 200", "category": "국내주식",
     "description": "KOSPI 200 지수를 추종하는 국내 대표 ETF. 삼성전자, SK하이닉스 등 대형주 중심.",
     "expense_ratio": 0.15, "aum_billion": 58000, "risk_level": "중간"},
    {"ticker": "TIGER미국S&P500", "name": "TIGER 미국 S&P500", "category": "해외주식",
     "description": "미국 S&P 500 지수를 추종. 애플, 마이크로소프트 등 미국 대형주 투자.",
     "expense_ratio": 0.07, "aum_billion": 45000, "risk_level": "중간"},
    {"ticker": "KODEX배당가치", "name": "KODEX 배당가치", "category": "배당",
     "description": "고배당 가치주 중심 ETF. 은행, 통신 등 배당 수익률이 높은 종목.",
     "expense_ratio": 0.30, "aum_billion": 8500, "risk_level": "낮음"},
    {"ticker": "TIGER반도체", "name": "TIGER 반도체", "category": "섹터",
     "description": "국내 반도체 산업 ETF. 삼성전자, SK하이닉스 등 반도체 관련주.",
     "expense_ratio": 0.40, "aum_billion": 12000, "risk_level": "높음"},
    {"ticker": "KODEX국고채3년", "name": "KODEX 국고채 3년", "category": "채권",
     "description": "한국 3년 만기 국고채에 투자하는 안정형 ETF.",
     "expense_ratio": 0.05, "aum_billion": 25000, "risk_level": "매우낮음"},
    {"ticker": "TIGER차이나CSI300", "name": "TIGER 차이나 CSI300", "category": "해외주식",
     "description": "중국 CSI 300 지수 추종. 상해/심천 대형주 투자.",
     "expense_ratio": 0.25, "aum_billion": 3500, "risk_level": "높음"},
    {"ticker": "KODEX골드선물", "name": "KODEX 골드선물(H)", "category": "원자재",
     "description": "금 선물 가격을 추종하는 원자재 ETF. 인플레이션 헤지 수단.",
     "expense_ratio": 0.68, "aum_billion": 5200, "risk_level": "중간"},
    {"ticker": "TIGER리츠부동산", "name": "TIGER 리츠부동산인프라", "category": "부동산",
     "description": "국내 리츠 및 부동산 인프라 기업에 투자. 배당 수익 추구.",
     "expense_ratio": 0.29, "aum_billion": 4100, "risk_level": "중간"},
    {"ticker": "KODEX2차전지", "name": "KODEX 2차전지산업", "category": "섹터",
     "description": "2차전지 관련 기업에 투자. LG에너지솔루션, 삼성SDI 등.",
     "expense_ratio": 0.45, "aum_billion": 18000, "risk_level": "높음"},
    {"ticker": "TIGER단기통안채", "name": "TIGER 단기통안채", "category": "채권",
     "description": "초단기 통안채에 투자. 파킹 용도로 활용되는 안전 자산.",
     "expense_ratio": 0.03, "aum_billion": 32000, "risk_level": "매우낮음"},
]

EVAL_QUERIES = [
    {"query": "안정적인 배당 ETF를 추천해주세요",
     "reference": "KODEX 배당가치 ETF를 추천합니다. 은행, 통신 등 고배당 가치주에 투자하며 리스크가 낮습니다. 수수료는 0.30%입니다.",
     "relevant_etfs": ["KODEX배당가치", "TIGER리츠부동산"]},
    {"query": "미국 주식에 투자하고 싶어요",
     "reference": "TIGER 미국 S&P500 ETF를 추천합니다. 애플, 마이크로소프트 등 미국 대형주에 투자하며 수수료가 0.07%로 저렴합니다.",
     "relevant_etfs": ["TIGER미국S&P500"]},
    {"query": "원금 손실 위험이 적은 ETF는?",
     "reference": "TIGER 단기통안채 ETF를 추천합니다. 초단기 통안채에 투자하여 원금 손실 위험이 매우 낮으며 수수료도 0.03%입니다.",
     "relevant_etfs": ["KODEX국고채3년", "TIGER단기통안채"]},
    {"query": "반도체 섹터에 투자하려면?",
     "reference": "TIGER 반도체 ETF를 추천합니다. 삼성전자, SK하이닉스 등 반도체 관련주에 집중 투자합니다. 리스크가 높으니 주의하세요.",
     "relevant_etfs": ["TIGER반도체"]},
    {"query": "인플레이션 헤지 방법이 있을까요?",
     "reference": "KODEX 골드선물(H) ETF를 고려해보세요. 금 선물 가격을 추종하여 인플레이션 헤지 수단으로 활용됩니다.",
     "relevant_etfs": ["KODEX골드선물"]},
    {"query": "중국 시장에 투자하는 ETF는?",
     "reference": "TIGER 차이나 CSI300 ETF가 있습니다. 상해/심천 대형주에 투자하지만 리스크가 높으니 신중하게 투자하세요.",
     "relevant_etfs": ["TIGER차이나CSI300"]},
]

print(f"✅ ETF {len(SAMPLE_ETF_DATA)}개, 평가 질의 {len(EVAL_QUERIES)}개")

## Weekend 1 복원: 벡터 스토어 + 하이브리드 검색

In [ ]:
# Weekend 1 체크포인트 복원
import pickle
from langchain.schema import Document
from rank_bm25 import BM25Okapi

vs_path = "project2_data/vectorstore/faiss_baseline"

if os.path.exists(vs_path):
    vs = FAISS.load_local(vs_path, embeddings, allow_dangerous_deserialization=True)
    with open("project2_data/raw/etf_documents.json") as f:
        etf_docs = json.load(f)
    print(f"✅ 체크포인트 복원: {vs.index.ntotal}개 문서")
else:
    docs = []
    etf_docs = {}
    for etf in SAMPLE_ETF_DATA:
        content = f"{etf['name']} ({etf['ticker']}): {etf['description']} 카테고리: {etf['category']}, 수수료: {etf['expense_ratio']}%, 리스크: {etf['risk_level']}"
        docs.append(Document(page_content=content, metadata={"ticker": etf["ticker"]}))
        etf_docs[etf["ticker"]] = {"name": etf["name"], "content": content, "category": etf["category"]}
    vs = FAISS.from_documents(docs, embeddings)
    print(f"✅ 새로 구축: {vs.index.ntotal}개 문서")

all_contents = [etf_docs[k]["content"] for k in etf_docs]
all_keys = list(etf_docs.keys())
bm25 = BM25Okapi([doc.split() for doc in all_contents])

def hybrid_search(query, k=5, alpha=0.5):
    """BM25 + 벡터 하이브리드 검색 (RRF)"""
    vec_results = vs.similarity_search_with_score(query, k=k)
    vec_ids = [(doc.metadata.get("ticker", ""), score) for doc, score in vec_results]
    bm25_scores = bm25.get_scores(query.split())
    bm25_ranked = sorted(enumerate(bm25_scores), key=lambda x: x[1], reverse=True)[:k]
    bm25_ids = [(all_keys[idx], score) for idx, score in bm25_ranked]
    rrf_scores = {}
    for rank, (doc_id, _) in enumerate(vec_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + alpha / (rank + 60)
    for rank, (doc_id, _) in enumerate(bm25_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1 - alpha) / (rank + 60)
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return [(doc_id, rrf_scores[doc_id], etf_docs.get(doc_id, {}).get("content", "")) for doc_id, _ in sorted_results]

print("✅ hybrid_search 준비 완료")

---
### 실습 1: Hit Rate 계산

`hybrid_search`의 Hit Rate@K를 계산하세요.

- `hit_rate(eval_data, search_fn, k)` 함수 구현
- `EVAL_QUERIES`의 `relevant_etfs` 중 하나라도 검색 결과에 있으면 hit
- k=3, k=5에서 각각 측정

In [ ]:
# 실습 1
def hit_rate(eval_data, search_fn, k=5):
    hits = 0
    for item in eval_data:
        results = search_fn(item["query"], k=k)
        found_ids = [r[0] for r in results]
        # ---- 여기에 코드 작성 ----
        pass
    return hits / len(eval_data)

for k in [3, 5]:
    hr = hit_rate(EVAL_QUERIES, hybrid_search, k=k)
    print(f"Hit Rate@{k}: {hr:.3f}")

---
### RAG 답변 생성 파이프라인

In [ ]:
# RAG 답변 생성
def ask_etf(query, k=3, verbose=False):
    results = hybrid_search(query, k=k)
    context = "\n\n".join([f"[{doc_id}] {content}" for doc_id, score, content in results])
    _llm = ChatOpenAI(model=MODEL, temperature=0)
    answer = _llm.invoke([
        SystemMessage(content="ETF 전문가입니다. 검색된 문서만을 근거로 답변하세요. 추측하지 마세요."),
        HumanMessage(content=f"참고 문서:\n{context}\n\n질문: {query}")
    ]).content
    if verbose:
        print(f"Q: {query}\nA: {answer[:200]}...")
    return answer

# 전체 답변 생성 (이후 실습에서 사용)
answers = {}
for item in EVAL_QUERIES:
    answers[item["query"]] = ask_etf(item["query"])
print(f"✅ {len(answers)}개 답변 생성 완료")

### 실습 2: temperature 비교

temperature=0 vs 0.7로 답변을 생성하고 길이, 소요시간을 비교하세요.

- `ask_etf_v2(query, k, temperature, max_tokens)` → `(answer, elapsed)` 반환
- `ChatOpenAI(model=MODEL, temperature=..., max_tokens=...).invoke()` 사용

In [ ]:
# 실습 2
def ask_etf_v2(query, k=3, temperature=0, max_tokens=500):
    results = hybrid_search(query, k=k)
    context = "\n".join([f"[{did}] {c}" for did, _, c in results])
    start = time.time()
    # ---- 여기에 코드 작성 ----
    pass

q = "안정적인 배당 ETF를 추천해주세요"
a0, t0 = ask_etf_v2(q, temperature=0)
a7, t7 = ask_etf_v2(q, temperature=0.7)
print(f"temp=0: {len(a0)}자, {t0:.2f}초")
print(f"temp=0.7: {len(a7)}자, {t7:.2f}초")

---
### LLM 리랭킹

In [ ]:
# LLM 기반 리랭킹
def llm_rerank(query, documents, top_k=3):
    doc_list = "\n".join([
        f"[{i}] {doc_id}: {content[:150]}"
        for i, (doc_id, score, content) in enumerate(documents)
    ])
    _llm = ChatOpenAI(model=MODEL, temperature=0).bind(
        response_format={"type": "json_object"}
    )
    response = _llm.invoke([
        SystemMessage(content="ETF 검색 결과를 질의 관련성 순으로 재정렬하는 전문가입니다."),
        HumanMessage(content=f"""질문: {query}

검색 결과:
{doc_list}

위 검색 결과를 질문과의 관련성 순으로 정렬하세요.
각 문서에 1-10점 관련성 점수를 부여하세요.
JSON: {{"rankings": [{{"index": 0, "score": 9, "reason": "이유"}}]}}""")
    ])
    try:
        result = json.loads(response.content)
        rankings = result if isinstance(result, list) else result.get("rankings", result.get("results", []))
        rankings.sort(key=lambda x: x.get("score", 0), reverse=True)
        reranked = []
        for r in rankings[:top_k]:
            idx = r["index"]
            if 0 <= idx < len(documents):
                doc_id, _, content = documents[idx]
                reranked.append((doc_id, r["score"], content))
        return reranked
    except Exception as e:
        print(f"⚠️ 리랭킹 실패: {e}")
        return documents[:top_k]

print("✅ llm_rerank 준비 완료")

### 실습 3: 리랭킹 전후 Hit Rate 비교

리랭킹 적용 전후의 Hit Rate를 비교하세요.

- `hybrid_search(k=7)` → 상위 `k_rerank`개 vs `llm_rerank()` 후 `k_rerank`개
- 쿼리별 결과와 평균 소요시간 출력

In [ ]:
# 실습 3
def compare_reranking(eval_data, k_initial=7, k_rerank=3):
    hit_before, hit_after = 0, 0
    total_time = 0

    for item in eval_data:
        initial = hybrid_search(item["query"], k=k_initial)
        ids_before = [r[0] for r in initial[:k_rerank]]

        start = time.time()
        reranked = llm_rerank(item["query"], initial, top_k=k_rerank)
        total_time += time.time() - start
        ids_after = [r[0] for r in reranked]

        # ---- 여기에 코드 작성 ----
        # ids_before, ids_after에서 relevant_etfs 존재 여부 확인
        pass

    n = len(eval_data)
    print(f"\nHit Rate@{k_rerank}: {hit_before/n:.3f} → {hit_after/n:.3f}")
    print(f"평균 리랭킹 시간: {total_time/n:.2f}초")

compare_reranking(EVAL_QUERIES)

---
### 스코어 필터링 + 통합 파이프라인

In [ ]:
# 스코어 필터링
def score_filter(results, method="dynamic"):
    if not results:
        return results
    scores = [r[1] for r in results]
    if method == "fixed":
        threshold = 5.0
    elif method == "dynamic":
        threshold = np.mean(scores) - np.std(scores)
    elif method == "gap":
        gaps = [scores[i] - scores[i+1] for i in range(len(scores)-1)]
        threshold = scores[np.argmax(gaps) + 1] + 0.01 if gaps else 0
    else:
        threshold = 0
    filtered = [(d, s, c) for d, s, c in results if s >= threshold]
    return filtered if filtered else results[:1]

def rag_pipeline(query, k_search=7, k_rerank=5, filter_method="dynamic"):
    initial = hybrid_search(query, k=k_search)
    reranked = llm_rerank(query, initial, top_k=k_rerank)
    filtered = score_filter(reranked, method=filter_method)
    context = "\n".join([f"[{did}] {c}" for did, _, c in filtered])
    answer = ChatOpenAI(model=MODEL, temperature=0).invoke([
        SystemMessage(content="ETF 전문가입니다. 검색된 문서만을 근거로 답변하세요."),
        HumanMessage(content=f"참고 문서:\n{context}\n\n질문: {query}")
    ]).content
    return {"query": query, "answer": answer,
            "retrieved": [r[0] for r in initial],
            "reranked": [r[0] for r in reranked],
            "filtered": [r[0] for r in filtered]}

print("✅ score_filter, rag_pipeline 준비 완료")

### 실습 4: 스코어 필터링 방법 비교

3가지 필터링(fixed, dynamic, gap)의 평균 통과 문서 수를 비교하세요.

- `hybrid_search()` → `llm_rerank()` → `score_filter()` 순서로 적용

In [ ]:
# 실습 4
for method in ["fixed", "dynamic", "gap"]:
    total_docs = 0
    for item in EVAL_QUERIES:
        initial = hybrid_search(item["query"], k=7)
        reranked = llm_rerank(item["query"], initial, top_k=5)
        # ---- 여기에 코드 작성 ----
        pass
    avg_docs = total_docs / len(EVAL_QUERIES)
    print(f"{method:8s}: 평균 {avg_docs:.1f}개 문서 통과")

---
### 프롬프트 최적화

In [ ]:
# 프롬프트 비교 (기본 vs 최적화)
basic_prompt = ChatPromptTemplate.from_messages([
    ("system", "ETF 전문가입니다."),
    ("human", "문서: {context}\n\n질문: {query}")
])

optimized_prompt = ChatPromptTemplate.from_messages([
    ("system", """10년 경력의 ETF 애널리스트입니다.
답변 규칙:
1. 검색된 문서 내용만을 근거로 답변
2. 추측 정보는 "확인이 필요합니다"라고 명시
3. 투자 위험을 반드시 고지
4. 구체적인 수치 포함

답변 형식:
## 추천 ETF
## 추천 이유
## 주의사항"""),
    ("human", "참고 문서:\n{context}\n\n질문: {query}")
])

query = "안정적인 배당 ETF를 추천해주세요"
results = hybrid_search(query, k=3)
context = "\n".join([f"[{did}] {c}" for did, _, c in results])

for name, prompt in [("기본", basic_prompt), ("최적화", optimized_prompt)]:
    msgs = prompt.format_messages(context=context, query=query)
    resp = llm.invoke(msgs)
    print(f"=== {name} ===\n{resp.content[:200]}\n")

### 실습 5: Few-shot 프롬프트 설계

ETF 추천 답변 예시를 포함한 Few-shot 프롬프트를 만들고 실행하세요.

- `ChatPromptTemplate.from_messages()`로 시스템 + 예시 포함
- 예시에 "추천 ETF / 추천 이유 / 주의사항" 구조 포함
- `llm.invoke()`로 실행

In [ ]:
# 실습 5
# ---- 여기에 코드 작성 ----
# fewshot_prompt = ChatPromptTemplate.from_messages([...])
# msgs = fewshot_prompt.format_messages(context=..., query=...)
# resp = llm.invoke(msgs)
# print(resp.content)

---
### BLEU Score 구현

In [ ]:
# BLEU Score
def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def modified_precision(ref_tokens, cand_tokens, n):
    ref_ngrams = Counter(get_ngrams(ref_tokens, n))
    cand_ngrams = Counter(get_ngrams(cand_tokens, n))
    clipped = sum(min(count, ref_ngrams.get(ng, 0)) for ng, count in cand_ngrams.items())
    total = sum(cand_ngrams.values())
    return clipped / total if total > 0 else 0.0

def brevity_penalty(ref_len, cand_len):
    return 1.0 if cand_len >= ref_len else math.exp(1 - ref_len / cand_len)

def compute_bleu(reference, candidate, max_n=4):
    ref_tokens, cand_tokens = reference.split(), candidate.split()
    bp = brevity_penalty(len(ref_tokens), len(cand_tokens))
    precisions = [modified_precision(ref_tokens, cand_tokens, n) for n in range(1, max_n + 1)]
    log_avg = 0.0
    for p in precisions:
        if p == 0:
            return 0.0
        log_avg += (1.0 / max_n) * math.log(p)
    return bp * math.exp(log_avg)

print("✅ compute_bleu 준비 완료")

### 실습 6: BLEU 스코어 계산

전체 평가 쿼리에 대해 BLEU-1, BLEU-2, BLEU-4를 계산하세요.

- `compute_bleu(ref, answer, max_n=N)` 사용
- DataFrame으로 정리, 평균 출력

In [ ]:
# 실습 6
rows = []
for item in EVAL_QUERIES:
    answer = answers.get(item["query"], ask_etf(item["query"]))
    ref = item["reference"]
    # ---- 여기에 코드 작성 ----
    pass

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\n평균: BLEU-1={df['BLEU-1'].mean():.4f}, BLEU-4={df['BLEU-4'].mean():.4f}")

---
### ROUGE Score 구현

In [ ]:
# ROUGE Score
def rouge_n(reference, candidate, n=1):
    ref_tokens, cand_tokens = reference.split(), candidate.split()
    ref_ngrams = Counter(get_ngrams(ref_tokens, n))
    cand_ngrams = Counter(get_ngrams(cand_tokens, n))
    overlap = sum(min(ref_ngrams[ng], cand_ngrams.get(ng, 0)) for ng in ref_ngrams)
    recall = overlap / sum(ref_ngrams.values()) if ref_ngrams else 0.0
    precision = overlap / sum(cand_ngrams.values()) if cand_ngrams else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"recall": recall, "precision": precision, "f1": f1}

def lcs_length(seq1, seq2):
    m, n = len(seq1), len(seq2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if seq1[i-1] == seq2[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

def rouge_l(reference, candidate):
    ref_tokens, cand_tokens = reference.split(), candidate.split()
    lcs = lcs_length(ref_tokens, cand_tokens)
    recall = lcs / len(ref_tokens) if ref_tokens else 0.0
    precision = lcs / len(cand_tokens) if cand_tokens else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"recall": recall, "precision": precision, "f1": f1}

print("✅ rouge_n, rouge_l 준비 완료")

### 실습 7: BLEU와 ROUGE 비교

BLEU-4, ROUGE-1, ROUGE-2, ROUGE-L을 계산하고 상관관계를 분석하세요.

- DataFrame으로 정리
- BLEU-4 vs ROUGE-1 상관계수 출력

In [ ]:
# 실습 7
rows = []
for item in EVAL_QUERIES:
    a = answers.get(item["query"], ask_etf(item["query"]))
    # ---- 여기에 코드 작성 ----
    pass

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nBLEU-4 vs ROUGE-1 상관계수: {df['BLEU-4'].corr(df['R-1']):.3f}")

---
### BERTScore 구현

In [ ]:
# BERTScore (OpenAI 임베딩 기반)
def get_embedding(text):
    return np.array(embeddings.embed_query(text))

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def simple_bertscore(reference, candidate):
    return cosine_sim(get_embedding(reference), get_embedding(candidate))

print("✅ simple_bertscore 준비 완료")

### 실습 8: BERTScore와 상관관계 분석

BLEU-4, ROUGE-1, BERTScore 3개 지표의 상관관계를 분석하세요.

- 전체 쿼리에 대해 3개 지표 계산
- 3쌍의 상관계수 출력

In [ ]:
# 실습 8
rows = []
for item in EVAL_QUERIES:
    a = answers.get(item["query"], ask_etf(item["query"]))
    # ---- 여기에 코드 작성 ----
    pass

df = pd.DataFrame(rows)
print(df.to_string(index=False))
# 3쌍 상관계수 출력

---
### 한국어 금융 토크나이저

In [ ]:
# 한국어 금융 도메인 토큰 분리기
def korean_financial_tokenize(text):
    words = text.split()
    tokens = []
    particles = ['으로', '에서', '부터', '까지', '에게',
                 '은', '는', '이', '가', '을', '를', '에', '의', '와', '과', '도', '만', '로']
    for word in words:
        m = re.match(r'^([\d,.]+)(%|억|만|원|개|배)(.*)$', word)
        if m:
            tokens.extend([m.group(1), m.group(2)])
            if m.group(3): tokens.append(m.group(3))
            continue
        m = re.match(r'^([A-Za-z0-9&]+)([가-힣]+)$', word)
        if m:
            tokens.append(m.group(1))
            rest = m.group(2)
            separated = False
            for p in sorted(particles, key=len, reverse=True):
                if rest == p:
                    tokens.append(p); separated = True; break
                if rest.endswith(p) and len(rest) > len(p):
                    tokens.extend([rest[:-len(p)], p]); separated = True; break
            if not separated: tokens.append(rest)
            continue
        separated = False
        for p in sorted(particles, key=len, reverse=True):
            if word.endswith(p) and len(word) > len(p):
                tokens.extend([word[:-len(p)], p]); separated = True; break
        if not separated: tokens.append(word)
    return tokens

test = "KODEX200은 수수료가 0.15%로 ETF를 추천합니다"
print(f"공백: {test.split()}")
print(f"금융: {korean_financial_tokenize(test)}")

### 실습 9: 전문 용어 보호 토크나이저

전문 용어(2차전지, S&P500 등)를 분리하지 않는 개선된 토크나이저를 만드세요.

- `FINANCE_TERMS` 집합에 보호할 용어 등록
- 매칭되면 그대로 보존, 아니면 기존 `korean_financial_tokenize()` 적용

In [ ]:
# 실습 9
FINANCE_TERMS = {"2차전지", "S&P500", "CSI300", "KOSPI200", "KODEX200"}

def korean_financial_tokenize_v2(text):
    words = text.split()
    tokens = []
    for word in words:
        # ---- 여기에 코드 작성 ----
        pass
    return tokens

test = "2차전지 ETF와 S&P500 ETF 비교"
print(f"기존: {korean_financial_tokenize(test)}")
print(f"개선: {korean_financial_tokenize_v2(test)}")

---
### LLM-as-Judge

In [ ]:
# LLM-as-Judge
def llm_judge(query, answer, context, criteria=None):
    if criteria is None:
        criteria = {
            "정확성": "답변이 참고 문서의 사실과 일치하는가? (1-5)",
            "관련성": "질문의 의도와 범위에 적합한 답변인가? (1-5)",
            "완전성": "핵심 정보(ETF명, 수수료, 리스크 등)를 포함하는가? (1-5)",
            "안전성": "투자 위험을 적절히 고지하고 있는가? (1-5)",
            "명확성": "이해하기 쉽고 구조적인가? (1-5)",
        }
    criteria_text = "\n".join([f"- {k}: {v}" for k, v in criteria.items()])
    _llm = ChatOpenAI(model=MODEL, temperature=0).bind(response_format={"type": "json_object"})
    response = _llm.invoke([
        SystemMessage(content="금융 ETF 답변 품질 평가 전문가입니다. JSON으로 응답하세요."),
        HumanMessage(content=f"""다음 ETF 추천 답변을 평가해주세요.

질문: {query}
참고 문서: {context[:500]}
답변: {answer}

평가 기준:
{criteria_text}

JSON: {{"scores": {{"정확성": 4, ...}}, "총점": 20, "피드백": "..."}}""")
    ])
    try:
        return json.loads(response.content)
    except:
        return {"error": "파싱 실패"}

print("✅ llm_judge 준비 완료")

### 실습 10: LLM-as-Judge 배치 평가

전체 평가 쿼리에 대해 LLM-as-Judge를 실행하고 기준별 평균을 계산하세요.

- `llm_judge(query, answer, context)` 호출
- 총점과 기준별 점수 출력, 기준별 평균 계산

In [ ]:
# 실습 10
all_judgments = []
for item in EVAL_QUERIES:
    a = answers.get(item["query"], ask_etf(item["query"]))
    results = hybrid_search(item["query"], k=3)
    ctx = "\n".join([c for _, _, c in results])
    # ---- 여기에 코드 작성 ----
    pass

# 기준별 평균 점수 계산

---
### Criteria 기반 다차원 평가

In [ ]:
# Criteria 기반 평가
def criteria_evaluation(query, answer, context):
    criteria = {
        "사실_정확성": {"desc": "답변이 참고 문서의 사실과 일치하는가?", "weight": 0.30},
        "투자_적합성": {"desc": "추천이 질문자의 투자 목적에 적합한가?", "weight": 0.25},
        "리스크_고지": {"desc": "투자 위험을 적절히 고지하고 있는가?", "weight": 0.20},
        "정보_완전성": {"desc": "수수료, 수익률 등 필요 정보가 포함되었는가?", "weight": 0.15},
        "표현_명확성": {"desc": "전문 용어를 이해하기 쉽게 설명하는가?", "weight": 0.10},
    }
    _llm = ChatOpenAI(model=MODEL, temperature=0).bind(response_format={"type": "json_object"})
    results = {}
    for name, info in criteria.items():
        response = _llm.invoke([
            SystemMessage(content="금융 답변 품질 평가 전문가입니다. JSON으로 응답하세요."),
            HumanMessage(content=f"""기준: {info['desc']}
질문: {query}
참고 문서: {context[:400]}
답변: {answer}
1-5점으로 평가. JSON: {{"score": 4, "reason": "이유"}}""")
        ])
        try:
            r = json.loads(response.content)
            results[name] = {"score": r["score"], "reason": r.get("reason", ""), "weight": info["weight"]}
        except:
            results[name] = {"score": 0, "reason": "파싱 실패", "weight": info["weight"]}
    weighted_total = sum(r["score"] * r["weight"] for r in results.values())
    return {"criteria": results, "weighted_score": weighted_total, "normalized": weighted_total / 5.0 * 100}

print("✅ criteria_evaluation 준비 완료")

### 실습 11: Criteria 평가 레이더 차트

5개 기준의 평균 점수를 레이더 차트로 시각화하세요.

- labels: 사실_정확성, 투자_적합성, 리스크_고지, 정보_완전성, 표현_명확성
- matplotlib polar subplot 사용

In [ ]:
# 실습 11
labels = ["사실_정확성", "투자_적합성", "리스크_고지", "정보_완전성", "표현_명확성"]
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

# ---- 여기에 코드 작성 ----
# 1) all_judgments에서 기준별 평균 점수 추출
# 2) ax.plot() + ax.fill()로 레이더 차트

---
### 종합 평가 파이프라인

In [ ]:
# 종합 평가 파이프라인
class ETFEvaluationPipeline:
    def __init__(self, thresholds=None):
        self.thresholds = thresholds or {"ROUGE-1": 0.3, "BERTScore": 0.7, "LLM_Judge": 15}
        self.results = []

    def evaluate_single(self, query, answer, reference, context):
        bleu4 = compute_bleu(reference, answer, max_n=4)
        r1 = rouge_n(reference, answer, 1)["f1"]
        r2 = rouge_n(reference, answer, 2)["f1"]
        rl = rouge_l(reference, answer)["f1"]
        bs = simple_bertscore(reference, answer)
        judge = llm_judge(query, answer, context)
        judge_total = judge.get("총점", 0)

        gate = {
            "ROUGE-1": r1 >= self.thresholds["ROUGE-1"],
            "BERTScore": bs >= self.thresholds["BERTScore"],
            "LLM_Judge": judge_total >= self.thresholds["LLM_Judge"],
        }
        if not gate.get("BERTScore"):
            status = "🔴 FAIL"
        elif not all(gate.values()):
            status = "🟡 WARN"
        else:
            status = "🟢 PASS"

        result = {"query": query[:30], "BLEU-4": bleu4, "ROUGE-1": r1, "ROUGE-2": r2,
                  "ROUGE-L": rl, "BERTScore": bs, "LLM_Judge": judge_total, "status": status}
        self.results.append(result)
        return result

    def evaluate_batch(self, eval_data, answer_fn):
        for item in eval_data:
            answer = answer_fn(item["query"])
            context = "\n".join([c for _, _, c in hybrid_search(item["query"], k=3)])
            self.evaluate_single(item["query"], answer, item["reference"], context)
        return self

    def summary(self):
        df = pd.DataFrame(self.results)
        print("=== 종합 평가 결과 ===")
        print(df[["query", "BLEU-4", "ROUGE-1", "BERTScore", "LLM_Judge", "status"]].to_string(index=False))
        print(f"\n🟢 PASS: {sum(1 for r in self.results if '🟢' in r['status'])}")
        print(f"🟡 WARN: {sum(1 for r in self.results if '🟡' in r['status'])}")
        print(f"🔴 FAIL: {sum(1 for r in self.results if '🔴' in r['status'])}")
        return df

print("✅ ETFEvaluationPipeline 준비 완료")

### 실습 12: 통합 평가 파이프라인 실행

`ETFEvaluationPipeline`으로 전체 평가를 실행하고 CSV로 저장하세요.

- `evaluate_batch()` → `summary()` → `.to_csv()`

In [ ]:
# 실습 12
pipeline = ETFEvaluationPipeline()
# ---- 여기에 코드 작성 ----

---
### Gradio 평가 대시보드

In [ ]:
# Gradio 기본 대시보드
import gradio as gr

def evaluate_query(query, reference=""):
    initial = hybrid_search(query, k=7)
    reranked = llm_rerank(query, initial, top_k=3)
    filtered = score_filter(reranked, method="dynamic")

    context = "\n".join([f"[{did}] {c}" for did, _, c in filtered])
    answer = ChatOpenAI(model=MODEL, temperature=0).invoke([
        SystemMessage(content="ETF 전문가입니다. 검색된 문서만을 근거로 답변하세요."),
        HumanMessage(content=f"참고 문서:\n{context}\n\n질문: {query}")
    ]).content

    search_info = "📋 검색 결과:\n" + "\n".join([f"  [{did}] score={score}" for did, score, _ in filtered])

    metrics_info = ""
    if reference.strip():
        b4 = compute_bleu(reference, answer, max_n=4)
        r1 = rouge_n(reference, answer, 1)["f1"]
        rl = rouge_l(reference, answer)["f1"]
        bs = simple_bertscore(reference, answer)
        metrics_info = f"BLEU-4: {b4:.4f} | ROUGE-1: {r1:.4f} | ROUGE-L: {rl:.4f} | BERTScore: {bs:.4f}"
    else:
        metrics_info = "참조 답변을 입력하면 자동 평가가 표시됩니다."

    judge = llm_judge(query, answer, context)
    judge_info = f"총점: {judge.get('총점', 'N/A')}/25 | 피드백: {judge.get('피드백', 'N/A')[:200]}"

    return answer, search_info, metrics_info, judge_info

print("✅ evaluate_query 준비 완료")

### 실습 13: 평가 대시보드 확장

기존 `evaluate_query`에 Criteria 평가와 품질 게이트를 추가하세요.

- `criteria_evaluation()` 결과 추가
- 출력에 Criteria 평가 Textbox 추가
- 품질 게이트: BERTScore >= 0.7 and LLM 총점 >= 15 → PASS

In [ ]:
# 실습 13
# ---- 여기에 코드 작성 ----
# 1) evaluate_query_extended 함수: criteria_evaluation 결과 추가
# 2) gr.Interface 에 Criteria 평가 출력 추가
# 3) 품질 게이트 판정

---
### 체크포인트 저장

In [ ]:
# Weekend 2 체크포인트 저장
os.makedirs("project2_data/checkpoints", exist_ok=True)

weekend2_results = {
    "timestamp": datetime.now().isoformat(),
    "evaluation_metrics": ["BLEU", "ROUGE", "BERTScore", "LLM-as-Judge", "Criteria"],
    "pipeline_config": {"search_k": 7, "rerank_k": 3, "filter_method": "dynamic", "model": MODEL},
    "quality_gate_thresholds": {"ROUGE-1": 0.3, "BERTScore": 0.7, "LLM_Judge": 15},
}
with open("project2_data/checkpoints/weekend2_results.json", "w") as f:
    json.dump(weekend2_results, f, ensure_ascii=False, indent=2)

with open("project2_data/checkpoints/weekend2_answers.json", "w") as f:
    json.dump(answers, f, ensure_ascii=False, indent=2)

weekend2_state = {
    "weekend": 2, "completed": True,
    "achievements": [
        "LLM 리랭킹 파이프라인", "스코어 필터링", "프롬프트 최적화",
        "BLEU/ROUGE/BERTScore 평가", "한국어 토큰화",
        "LLM-as-Judge", "Criteria 평가", "종합 파이프라인 + 품질 게이트",
    ]
}
with open("project2_data/checkpoints/weekend2_state.json", "w") as f:
    json.dump(weekend2_state, f, ensure_ascii=False, indent=2)

print("✅ Weekend 2 체크포인트 저장 완료")

### 실습 14: 통합 리포트 + 체크리스트

Weekend 1+2 통합 리포트를 JSON으로 저장하고 필수 파일 체크리스트를 확인하세요.

- `report` 딕셔너리에 weekend1, weekend2 정보 통합
- 벡터 스토어, ETF 문서, 결과 파일 존재 여부 체크리스트 출력

In [ ]:
# 실습 14
report = {
    "weekend1": {"methods": ["FAISS", "BM25", "Hybrid"], "best": "Hybrid"},
    "weekend2": weekend2_state,
}
# ---- 여기에 코드 작성 ----